In [ ]:
# train.py
import os
import json
from dataclasses import dataclass
from typing import List, Dict, Any
from functools import partial

import numpy as np
from datasets import load_dataset, Dataset, concatenate_datasets
from torch.utils.data import Dataset 
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    Trainer,
AutoModelForCausalLM
)
from transformers.trainer_utils import get_last_checkpoint
from accelerate import Accelerator
import torch
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import torch.nn as nn
import pandas as pd

import torch

from dataclasses import dataclass
from typing import Dict, Any, Callable, Optional
import evaluate
import shutil
import glob
from datetime import datetime
from peft import LoraConfig, get_peft_model

import os
import argparse
import numpy as np
import torch
from datetime import datetime

from transformers import AutoTokenizer, TrainingArguments, Trainer
import sys
from pathlib import Path
import wandb

In [ ]:
odf = pd.read_csv("full_babe_with_cats.csv")
odf['text'] = odf['text'].astype(str).fillna("")
odf["followed_news_outlets_clean"] = (
    odf["followed_news_outlets"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace('"', "", regex=False)
)
odf["num_foll_news_outlet"] = odf["followed_news_outlets_clean"].apply(
    lambda x: len([item for item in x.split(",") if item.strip() != ""])
)

odf["num_age"] = pd.cut(odf["age"],bins=[-np.inf, 18, 30, 45, 55, 65, np.inf],labels=False)

odf["num_poli_idealogy"] = pd.cut(odf["political_ideology"],
                                 bins=[-np.inf, -6, -1, 1, 5, np.inf],
                                 labels=False)

odf["num_foll_news_outlet"] = pd.cut(odf["num_foll_news_outlet"],
                                          bins=[-np.inf, 1, 2, 3, 4, 6, np.inf],
                                          labels=False)


odf["num_news_check_frequency"] = odf["news_check_frequency"].map({
    "Never": 0,
    "Very rarely": 1,
    "Several times per month": 2,
    "Several times per week": 3,
    "Every day": 4,
    "Several times per day": 5,
})

def bin_education(edu):
    edu = str(edu).strip()
    if edu in {"Graduate work"}:
        return 5
    if edu in {"BA", "Bachelor", "Bachelors", "Bachelor's degree", "Bachelor’s degree"}:
        return 4
    if edu in {"Associate degree"}:
        return 3
    if edu in {"Some college", "Vocational or technical school"}:
        return 2
    if edu == "High school graduate":
        return 1
    return 0

odf["num_education"] = odf["education"].apply(bin_education)
odf['num_gender'] = odf['gender'].map({
    'Male': 1, 'male': 1,
    'Female': 0, 'female': 0,
}).fillna(2).astype(int)
odf["label_binary"] = odf["label_bias"].str.startswith("Biased").astype(int)


In [ ]:
#embeds

cat_cardinalities = {
    "words": 11267,
    "factual": 3,
    "group_id": 85,
    "type": 3,
    "topic": 14,
    "outlet": 8,
    "age": 54,
    "gender": 3,
    "education": 8,
    "native_english_speaker": 3,
    "political_ideology": 21,
    "followed_news_outlets": 551,
    "news_check_frequency": 6,
    "num_foll_news_outlet": 6,
    "num_poli_idealogy": 6,
    "num_age": 6,
    "num_gender": 3,
    "num_education": 6,
}


out_cats = ["num_age", "num_gender", "num_education", "num_poli_idealogy", "num_news_check_frequency", "num_foll_news_outlet"]
in_cats = [k for k, card in cat_cardinalities.items() if k not in ["num_age", "num_gender", "num_education", "num_poli_idealogy", "num_news_check_frequency", "num_foll_news_outlet"]]
cat_embs_info = {}
for k, card in cat_cardinalities.items():
    if k not in odf.columns:
        continue
    emb_dim = max(2, int(card ** 0.5) + 1) #sqrt(cardinality)) + 1
    emb_dim = min(50, emb_dim) #but not more than 50 - cough cough words
    cat_embs_info[k] = {
        "cardinality": card, 
        "emb_dim": emb_dim,
        "out_cat": k in out_cats,
        "in_cat": k in in_cats}




class MTLDataset(Dataset):
    def __init__(self, 
                 df, 
                 cat_info, 
                 tokenizer,
                 zero_cats = False,
                 max_length=128):
        
        df = df.reset_index(drop=True).copy()
        self.texts = df["text"].tolist()
     
        for col in list(cat_info.keys()):
            df[col] = df[col].astype('category')
            try:
                df[col] = df[col].cat.codes
            except:
                print(col)

        cat_ids = df[list(cat_info.keys())].values   #[N, num_cat_vars]
        cat_ids = cat_ids.astype(int)

        if zero_cats:
            cat_ids = np.zeros_like(cat_ids)

    
        self.cat_ids = cat_ids
        self.labels = df["label_binary"].to_numpy(dtype=np.float32)
        self.out_cat_names = [k for k, v in cat_info.items() if v.get("out_cat", False)]
        self.out_cat_ids = {name: df[name].values for name in self.out_cat_names}
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        cats = self.cat_ids[idx]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "x_cats": torch.tensor(cats, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
            "target_cats": torch.tensor(
                [self.out_cat_ids[name][idx] for name in self.out_cat_names],
                dtype=torch.long
            ),
        }
        return item
    
    


In [ ]:
class ambiProbe(nn.Module):

    
    def __init__(self, 
                cats_info, 
                llm_pretrained_name
                ):
        super().__init__()

        self.llm = AutoModel.from_pretrained(
            llm_pretrained_name,
            output_hidden_states=True
        )
        for p in self.llm.parameters():
            p.requires_grad = False
        
        self.llm.eval()
                

        d_model = self.llm.config.hidden_size

        late_fusion_width = d_model + 1 #plus 1 for the label
        self.linear_post_fusion = nn.Linear(late_fusion_width, d_model)

        
        self.shared_trunk = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model // 4),
        )
        
        self.out_cat_names = [k for k, v in cats_info.items() if v.get("out_cat", False)]
        self.probe_heads = nn.ModuleDict({
            name: nn.Linear(d_model // 4, cats_info[name]["cardinality"])
            for name in self.out_cat_names
        })

    def forward(
        self,
        input_ids, #embedding of input text
        attention_mask,
        x_cats,
        labels, # [B] binary silver labels
        target_cats  # [B, out_cats] numerical features
    ):

        outputs = self.llm(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        #uncollate here instead of building a collator for the dataset?
        target_cats_dict = {}
        for idx, name in enumerate(self.out_cat_names):
            target_cats_dict[name] = target_cats[:, idx]

        last_hidden = outputs.last_hidden_state # [B, L, d_model]
        mask = attention_mask.unsqueeze(-1)            # [B, L, 1]
        hidden_masked = last_hidden * mask             # [B, L, d_model]
        sum_hidden = hidden_masked.sum(dim=1)          # [B, d_model]
        lengths = mask.sum(dim=1).clamp(min=1)         # [B, 1]
        h_text = sum_hidden / lengths                  # [B, d_model]
        

        h_fused = torch.cat([h_text, labels.unsqueeze(-1).float()], dim=-1) # [B, d_model + 1]
        h_proj = self.linear_post_fusion(h_fused)    # [B, d_model]
        h_shared = self.shared_trunk(h_proj)         # [B, shared_hdim]

        

        logits = {name: head(h_shared) for name, head in self.probe_heads.items()}

        for name in self.probe_heads:
            target = target_cats_dict[name]
            logit = logits[name]
            n_classes = logit.shape[-1]


            bad = (target < 0) | (target >= n_classes)
            if bad.any():
                bad_vals = target[bad].detach().cpu().tolist()[:20]
                raise ValueError(
                    f"Bad targets for head {name}: {bad_vals}; "
                    f"valid range is [0, {n_classes - 1}]"
                )


                
        losses = {name: F.cross_entropy(logits[name], target_cats_dict[name]) 
                  for name in self.probe_heads}

        total_loss = torch.stack(list(losses.values())).sum()
        

        return {
            "loss": total_loss,
            "piecewise_losses": losses,
            "out_logits" : logits
        }
        

# TRAINING
#### Tiny Llama

In [ ]:
PROBE_WEIGHTS_DIR = './model/probes/'
run_name   = "TinyLlama_probe"
output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

wandb.init(
    project="bias-ambivalence-mtl",
    name=run_name,
    config={},
)
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

if os.exists('./model/probes/TinyLlama_probe/model.safetensors'):
    tl_probe = ambiProbe(cat_embs_info, "TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    from safetensors.torch import load_file
    tl_probe.load_state_dict(load_file('./model/probes/TinyLlama_probe/model.safetensors'))
    tl_probe.to(device)  # if you're on GPU
    tl_probe.eval()
else:
    tl_probe = ambiProbe(cat_embs_info, "TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    # ffdf = odf.sample(1000)
    ffdf = odf.copy()
    train_df, test_df = train_test_split(
        ffdf, 
        test_size=0.8, 
        random_state=42, 
        shuffle=True
    )
    train_dataset = MTLDataset(train_df, cat_embs_info, tokenizer, 
                               max_length=128, zero_cats=False)
    eval_dataset  = MTLDataset(test_df, cat_embs_info, tokenizer, 
                               max_length=128, zero_cats=False)
    
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        learning_rate=5e-5,
        num_train_epochs=3,
        weight_decay=0.01,
        save_total_limit=1,
        # report_to='wandb',
        seed=42,
    )
    
    trainer = Trainer(
        model=tl_probe,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        compute_metrics=None,
    )
    
    # trainer.train()
    # trainer.save_model(output_dir)

#### Qwen

In [ ]:
PROBE_WEIGHTS_DIR = './model/probes/'
run_name   = "Qwen_probe"
output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

wandb.init(
    project="bias-ambivalence-mtl",
    name=run_name,
    config={},
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-7B")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
model = ambiProbe(cat_embs_info, "Qwen/Qwen-7B")
# ffdf = odf.sample(1000)
ffdf = odf.copy()
train_df, test_df = train_test_split(
    ffdf, 
    test_size=0.8, 
    random_state=42, 
    shuffle=True
)
train_dataset = MTLDataset(train_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)
eval_dataset  = MTLDataset(test_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)


training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    # report_to='wandb',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=None,
)

trainer.train()
trainer.save_model(output_dir)

#### Deberta

In [ ]:
PROBE_WEIGHTS_DIR = './model/probes/'
run_name   = "magpie-pt_probe"
output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

wandb.init(
    project="bias-ambivalence-mtl",
    name=run_name,
    config={},
)
tokenizer = AutoTokenizer.from_pretrained("mediabiasgroup/magpie-pt")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
model = ambiProbe(cat_embs_info, "mediabiasgroup/magpie-pt")
# ffdf = odf.sample(1000)
ffdf = odf.copy()
train_df, test_df = train_test_split(
    ffdf, 
    test_size=0.8, 
    random_state=42, 
    shuffle=True
)
train_dataset = MTLDataset(train_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)
eval_dataset  = MTLDataset(test_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)


training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    # report_to='wandb',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=None,
)

trainer.train()
trainer.save_model(output_dir)
   

# TESTING
#### Tiny Llama

In [ ]:
import torch
from collections import Counter
from torch.utils.data import DataLoader
from tqdm import tqdm

model.eval()
device = next(model.parameters()).device
out_cat_names = eval_dataset.out_cat_names

marginal_acc = {}
for name in out_cat_names:
    counts = Counter(eval_dataset.out_cat_ids[name].tolist())
    marginal_acc[name] = max(counts.values()) / sum(counts.values())

correct = {name: 0 for name in out_cat_names}
total = 0
loader = DataLoader(eval_dataset, batch_size=64, shuffle=False)

with torch.no_grad():
    for batch in tqdm(loader, desc="Evaluating probe"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        x_cats         = batch["x_cats"].to(device)
        target_cats    = batch["target_cats"].to(device)
        
        out = model(input_ids, attention_mask, x_cats, labels, target_cats)
        
        bsz = input_ids.size(0)
        total += bsz
        for cat_idx, name in enumerate(out_cat_names):
            preds = out["out_logits"][name].argmax(dim=-1)
            golds = target_cats[:, cat_idx]
            correct[name] += (preds == golds).sum().item()

print(f"\n{'Variable':<25} {'Probe':>8}  {'Marginal':>10}  {'Lift':>8}")
print("─" * 55)
for name in out_cat_names:
    probe_acc = correct[name] / total
    marg_acc  = marginal_acc[name]
    lift      = probe_acc - marg_acc
    print(f"{name:<25} {probe_acc:>8.3f}  {marg_acc:>10.3f}  {lift:>+8.3f}")

print("─" * 55)
global_probe = sum(correct.values()) / (total * len(out_cat_names))
global_marg  = sum(marginal_acc.values()) / len(out_cat_names)
print(f"{'GLOBAL':<25} {global_probe:>8.3f}  {global_marg:>10.3f}  {global_probe - global_marg:>+8.3f}")

In [ ]:
import torch
import torch.nn.functional as F
import random

def bar(p, width=20):
    filled = int(round(p * width))
    return "█" * filled + "·" * (width - filled)

model.eval()
device = next(model.parameters()).device
out_cat_names = eval_dataset.out_cat_names
indices = random.sample(range(len(eval_dataset)), 3)

for idx in indices:
    item = eval_dataset[idx]
    text = eval_dataset.texts[idx]
    
    input_ids     = item["input_ids"].unsqueeze(0).to(device)
    attention_mask = item["attention_mask"].unsqueeze(0).to(device)
    labels        = item["labels"].unsqueeze(0).to(device)
    x_cats        = item["x_cats"].unsqueeze(0).to(device)
    target_cats   = item["target_cats"].unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = model(input_ids, attention_mask, x_cats, labels, target_cats)
    
    text_display = text if len(text) <= 140 else text[:140] + "…"
    print("\n" + "═" * 75)
    print(f"  Text:        {text_display}")
    print(f"  Bias label:  {int(item['labels'].item())}")
    print("─" * 75)
    
    for cat_idx, cat_name in enumerate(out_cat_names):
        logits = out["out_logits"][cat_name].squeeze(0)
        probs = F.softmax(logits, dim=-1)
        gold = int(item["target_cats"][cat_idx].item())
        pred = int(probs.argmax().item())
        correct = pred == gold
        
        flag = "✓" if correct else "✗"
        print(f"\n  {cat_name}   gold={gold}  pred={pred}  p(pred)={probs[pred].item():.2f}  p(gold)={probs[gold].item():.2f}  {flag}")
        
        for j, p in enumerate(probs.tolist()):
            if j == gold and j == pred:
                marker = "← gold/pred"
            elif j == gold:
                marker = "← gold"
            elif j == pred:
                marker = "← pred"
            else:
                marker = ""
            print(f"    {j}  {p:.2f}  {bar(p)}  {marker}")
    print()